# CS-781 Phase 3 — Rhetorical Role prediction (notebook G)

Same structure as the Phase-2 notebooks: your Phase-1 models + **your** N6 code (the head, LoRA / LoRMA / BitFit,
distillation — copied unchanged from Phase 2), a training script, and a **lab** that turns every finished run into
many Codabench zips plus a score chart.

**How a sentence gets a label.** Each document's sentences are tokenised with your Phase-1 tokenizer and encoded
together in windows (`scope: "document"`: every sentence sees its neighbours) or one by one (`scope: "sentence"`:
no context). Your `TokenClassifier` gives every token 13 role scores; the tokens of a sentence carry the sentence's
label in training (`label_tokens`), and the token scores are pooled into one score per sentence (mean / first token /
last token).

**The lab (no training)** works on the saved sentence scores of every run: each pooling × `TAUS` (prior correction:
rare roles pushed up — macro-F1 weighs every role equally; Menon et al. 2021) × `LAMBDAS` (Viterbi over the sentence
sequence with label-transition probabilities counted on the training labels — roles come in blocks); your
`PROB_ENSEMBLES` and greedy ensembles (Caruana et al. 2004). Every set is scored on the **official CL + IT dev
documents** with the official metric; `submissions/index.csv` + `submissions/score_chart.png` list them;
**you choose what to upload**. Settings marked YOUR CHOICE are yours (rule §17.5); declare in your LLM declaration
that the pipeline / lab / chart code was written with an assistant.

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** · Add-ons → Secrets → **`Capstone_Phase_1` attached**.

**Inputs (Add Input):** `notebook857159535b` (tokenizer) · your Phase-1 checkpoints: `phase1-continue` (decoder),
`Phase1_Training1` (encdec, swa), dataset `prefix`, `phase1-train-extra` (diffusion, moe). Later sessions: also this
notebook's own output, so earlier runs join the lab without retraining.

**How to use:** Save & Run All (~1.5 h for the 7 starting runs). Then Output → `submissions/`: `index.csv`,
`score_chart.png` and the zips. N8 checks the data first — if it fails, paste its error.

## N1. Login + download the Phase 3 data

In [ ]:
import os, shutil, json
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
os.makedirs("/kaggle/working/p3data", exist_ok=True)
for path in ["Train/CL_train.json", "Train/CL_dev.json", "Train/IT_train.json", "Train/IT_dev.json",
             "Train/label_vocab.json", "Test/CL_test.json", "Test/IT_test.json"]:
    src = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_3_RR/{path}", repo_type="dataset", token=HF_TOKEN)
    shutil.copy(src, f"/kaggle/working/p3data/{os.path.basename(path)}")
print(sorted(os.listdir("/kaggle/working/p3data")))

## N2. GPU check

In [ ]:
import torch
print("GPU count:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## N3. Configuration — experiments, RR options, lab settings

In [ ]:
import json
# =====================================================================================================
#  N3 — CONFIGURATION. Values marked YOUR CHOICE are yours (course rule §17.5); the comments explain each one.
# =====================================================================================================
#
#  PHASE 3 RULES THIS CONFIG MUST SATISFY (§9)
#  * all FOUR required Phase-1 architectures used: decoder, prefix (hybrid), encdec, swa (efficient);
#    the best four submissions come from DIFFERENT Phase-1 models
#  * ≥ 4 substantially different experiments: ≥ 3 different techniques FROM THE §9.2.1 LIST (LoRA, QLoRA, AdaLoRA,
#    rsLoRA, LORMA, Prefix Tuning, Prompt Tuning, IA³, KD) — LORMA COMPULSORY — and ≥ 1 knowledge distillation.
#    Full fine-tuning and BitFit are NOT on that list: you need one more listed technique besides LoRA + LoRMA + KD
#    (N6 has stubs for rsLoRA and IA³ for you to write).
#  * experiments on both domains (CL and IT) — every run here trains on both and reports both.
#
#  The experiments below are your Phase-2 configurations, unchanged, as the starting plan.

CONFIG = {
    "seed": 42,

    # ---------------- where each Phase-1 model's checkpoint is (searched under /kaggle/input) ----------------
    "checkpoints": {
        "decoder":   "continued/best_continued.pt",
        "prefix":    "prefix/best.pt",
        "encdec":    "encdec/best.pt",
        "swa":       "swa/best.pt",
        "diffusion": "diffusion/best.pt",
        "moe":       "moe/best.pt",
    },

    # ---------------- defaults for every experiment (an experiment below can override any of them) ----------------
    "defaults": {
        # --- RR-specific: how a sentence is seen and how it gets one label ---
        "scope": "document",     # "document": the document's sentences are encoded together in windows, so every
                                 #             sentence sees its neighbours (document context);
                                 # "sentence": every sentence is encoded alone (no context) — the brief asks you to
                                 #             compare sentence-only vs context-aware modelling.
        "label_tokens": "all",   # which tokens carry the sentence's label in training:
                                 #   "all"   every token of the sentence
                                 #   "last"  only the last token (a causal model has read the whole sentence there)
                                 #   "first" only the first token
        "pool": "mean",          # how the run's OWN zip (and its best-epoch choice) turns token scores into one
                                 # label per sentence: "mean" of the tokens' log-probs, or the "first" / "last"
                                 # token's. The lab (N11) tries every pooling the run supports anyway.
        # --- the rest as in Phase 2 ---
        "window": 1024,      # tokens per window (capped at what the model accepts: encdec 768)
        "stride": 768,       # 256 tokens of overlap; each token's scores come from the window where it is most central
        "context": "plain",  # "plain" or "echo" (causal models: the window is fed twice, the second copy is read)
        "epochs": 10,
        "patience": 3,
        "lr": 5e-5,          # YOUR CHOICE
        "batch_size": 8,     # windows per step (sentence scope packs short sentences up to batch_size x window tokens)
        "warmup_frac": 0.1,
        "weight_decay": 0.01,
        "grad_clip": 1.0,
        "head_dropout": 0.1,
        "class_weights": "none",   # "none" or "inv_sqrt" (rare roles cost more: weight ∝ 1/sqrt(count))
        "method_cfg": {},
        "moe_aux_coef": 0.0,
        "train_on_all": False,     # True = train on train + dev of both domains (no dev scores; fixed epochs)
        "epochs_from": None,       # for train_on_all runs: a finished dev-scored run whose best epoch is reused
        "teacher": None,
        "kd": {"temperature": 2.0, "alpha": 0.5},
        "student_layers": None,
        # "seed": an experiment may set its own seed (ensembles of the same settings with different seeds)
    },

    # ---------------- the experiments — YOUR CHOICE (starting plan = your Phase-2 settings) ----------------
    "experiments": {
        "prefix_full":    {"arch": "prefix", "method": "full"},
        "encdec_lorma":   {"arch": "encdec", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "decoder_lora":   {"arch": "decoder", "method": "lora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "swa_bitfit":     {"arch": "swa", "method": "bitfit", "lr": 1e-3},
        "diffusion_full": {"arch": "diffusion", "method": "full"},
        "moe_full":       {"arch": "moe", "method": "full"},
        "kd_moe_to_decoder4": {"arch": "decoder", "method": "full", "teacher": "moe_full", "student_layers": 4},
        # --- YOUR NEW TECHNIQUES (§9.2.1) — they run once you have written them in N6 section 6 ---------
        # Settings = your decoder_lora settings (r, alpha, lr); change them if you want (your choice).
        # Not written yet -> these two runs stop with NotImplementedError; the other runs are not affected.
        "decoder_rslora": {"arch": "decoder", "method": "rslora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "decoder_ia3":    {"arch": "decoder", "method": "ia3", "lr": 5e-4},
        # --- more of your own, e.g. (syntax only):
        # "name_sent": {<settings of another run>, "scope": "sentence"},      <- context ablation
        # "name_all": {<settings of a dev-scored run>, "train_on_all": True, "epochs_from": "<that run>"},
    },
}

# ---------------- which experiments THIS session trains (GPU) and smoke-tests (CPU) ----------------
SESSION_EXPERIMENTS = ["decoder_rslora", "decoder_ia3",             # your new methods first (fail fast if unwritten)
                       "moe_full", "decoder_lora", "kd_moe_to_decoder4", "encdec_lorma", "swa_bitfit",
                       "prefix_full", "diffusion_full"]     # a distillation student waits for its teacher
SMOKE_EXPERIMENTS = list(SESSION_EXPERIMENTS)              # a student's teacher is smoke-tested first automatically

# ---------------- lab: post-processing + ensembles of finished runs (N11) — YOUR CHOICE ----------------
LAMBDAS = [0.0, 0.5, 1.0]    # weight of the label-transition scores (Viterbi over the sentence sequence); 0 = off
TAUS = [0.0, 0.5, 1.0]       # prior correction strength (pushes rare roles up; macro-F1 weighs every role equally)
PROB_ENSEMBLES = {}          # {"name": ["run_a", "run_b"]} -> every pooling x lambda x tau variant, dev-scored;
                             # {"name": {"members": [...], "pool": "mean", "lam": 0.5, "tau": 0.5}} -> that one variant
GREEDY = True                # also build ensembles by greedy forward selection on dev
RUN_DIR_OVERRIDES = {}       # {"run": "/kaggle/input/<...>/rr_runs/run"} if a run is found twice and the wrong copy wins
TOP_N_ZIPS = 30              # zips written: this many best-on-dev sets + the best set of each run / ensemble

json.dump(CONFIG, open("/kaggle/working/run_config_rr.json", "w"), indent=1)
for name, e in CONFIG["experiments"].items():
    assert e["arch"] in CONFIG["checkpoints"], f"{name}: unknown arch {e['arch']}"
    t = e.get("teacher")
    assert not t or t in CONFIG["experiments"], f"{name}: teacher {t} is not an experiment"
    f = e.get("epochs_from")
    assert not f or (e.get("train_on_all") and f in CONFIG["experiments"]), f"{name}: epochs_from needs train_on_all and a known run"
for name in SESSION_EXPERIMENTS:
    assert name in CONFIG["experiments"], f"SESSION_EXPERIMENTS: {name} is not defined"
n = len(SESSION_EXPERIMENTS)
print(f"{len(CONFIG['experiments'])} experiments defined; this session trains {n}: {SESSION_EXPERIMENTS}")
if n:
    print(f"rough GPU time: ~{n} x 8-15 min, two at a time on T4 x2 -> about {n * 12 / 2 / 60:.1f} h")

## N4. Pipeline code: data, tokenisation, windows, pooling, decoding, metrics, backbones, files

In [ ]:
%%writefile rr_common.py
"""Phase 3 (Rhetorical Roles) pipeline: data of both domains (CL, IT), Phase-1 tokenisation of every sentence,
documents as token sequences with the sentence index of every token, windows, token labels, sentence pooling,
metrics (same rules as evaluate_phase3.py), sentence-level decoding (label-transition Viterbi, prior correction),
Phase-1 backbones that return hidden states, finder for finished runs, prediction and submission files."""
import glob, json, math, os, pickle, sys, zipfile
from collections import Counter
import numpy as np, torch, torch.nn as nn

INPUT_ROOT = os.environ.get("P3_INPUT", "/kaggle/input")      # read-only inputs (data prep, Phase-1 checkpoints)
WORK = os.environ.get("P3_WORK", "/kaggle/working")           # everything this notebook writes
DATA = f"{WORK}/p3data"                                        # Phase-3 files downloaded by cell N1
DOMAINS = ("CL", "IT")

def find_input(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{pattern}' not found under {INPUT_ROOT} — is the right notebook/dataset added as input?")
    return hits[0]

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

# ------------------------------------------------------------------------------------------------ labels
def _load_vocab():
    """label_vocab.json as a list or a mapping (same rules as evaluate_phase3.py) -> (labels, id->label)"""
    v = json.load(open(f"{DATA}/label_vocab.json", encoding="utf-8"))
    if isinstance(v, list):
        labels, id2 = list(v), dict(enumerate(v))
    elif all(isinstance(x, int) and not isinstance(x, bool) for x in v.values()):      # label -> id
        labels, id2 = list(v), {i: l for l, i in v.items()}
    else:                                                                              # id -> label
        labels, id2 = list(v.values()), {int(k): l for k, l in v.items()}
    assert len(labels) == 13 and len(set(labels)) == 13, f"expected 13 unique labels, got {labels}"
    return labels, id2

LABELS, ID2LABEL = _load_vocab()
L2I = {l: i for i, l in enumerate(LABELS)}
IGNORE = -100

# ------------------------------------------------------------------------------------------------ data
def _as_label(x):
    if isinstance(x, str) and x in L2I:
        return x
    if isinstance(x, (int, np.integer)) or (isinstance(x, str) and x.isdigit()):
        return ID2LABEL[int(x)]
    raise ValueError(f"unknown label {x!r} (labels: {LABELS})")

def _docs(obj, domain, split):
    """Accepts the documented format (a list of {"id", "text": [sentences], "labels": [...]}) and close variants:
    a dict {id: {...}} or {id: [sentences]}, "sentences" instead of "text", integer label ids."""
    items = list(obj.items()) if isinstance(obj, dict) else [(None, x) for x in obj]
    out = []
    for k, it in items:
        if isinstance(it, list):
            it = {"text": it}
        doc_id = it.get("id", it.get("doc_id", k))
        sents = it.get("text", it.get("sentences"))
        labs = it.get("labels", it.get("label"))
        if doc_id is None or not isinstance(sents, list):
            raise ValueError(f"{domain}_{split}: cannot read a document (keys: {list(it)[:8]}). Expected "
                             '{"id": ..., "text": [sentence, ...], "labels": [...]}')
        sents = [s if isinstance(s, str) else str(s) for s in sents]
        if labs is not None:
            labs = [_as_label(x) for x in labs]
            assert len(labs) == len(sents), f"{domain}_{split} {doc_id}: {len(sents)} sentences but {len(labs)} labels"
        out.append({"id": str(doc_id), "domain": domain, "key": f"{domain}|{doc_id}", "sents": sents, "labels": labs})
    return out

def load_data():
    """-> train, dev, test: lists of documents {"id", "domain", "key", "sents", "labels" (None for test)}"""
    def split(name):
        docs = []
        for dom in DOMAINS:
            docs += _docs(json.load(open(f"{DATA}/{dom}_{name}.json", encoding="utf-8")), dom, name)
        return docs
    return split("train"), split("dev"), split("test")

# ------------------------------------------------------------------------------------------------ tokenisation
_BPE = None
def bpe_tok_bpe():
    global _BPE
    if _BPE is None:
        _BPE = bpe_tok.load_bpe(find_input("bpe.json"))
    return _BPE

class SentTokenizer:
    """Your Phase-1 tokenisation (bpe_tok.FastEncoder, exactly as in pretraining), sentence by sentence.
    doc(sents) -> token ids of the whole document + the sentence index of every token. An empty sentence gets one
    <unk> token, so every sentence has at least one token (and therefore a prediction)."""
    def __init__(self):
        self.enc = bpe_tok.FastEncoder(bpe_tok_bpe())
        self.unk = bpe_tok_bpe().stoi["<unk>"]

    def doc(self, sents):
        ids, sent_of = [], []
        for j, s in enumerate(sents):
            t = self.enc.encode(s) or [self.unk]
            ids += t; sent_of += [j] * len(t)
        return ids, np.array(sent_of, dtype=np.int64)

def sentence_bounds(sent_of, n_sents):
    """first and last token index of every sentence"""
    first = np.searchsorted(sent_of, np.arange(n_sents), side="left")
    last = np.searchsorted(sent_of, np.arange(n_sents), side="right") - 1
    return first, last

def token_labels(sent_of, labels, mode="all"):
    """label id per token. all: every token of a sentence carries the sentence's label;
    first / last: only the sentence's first / last token does (the others are ignored in the loss)."""
    y = np.array([L2I[labels[s]] for s in sent_of], dtype=np.int64)
    if mode == "all":
        return y
    first, last = sentence_bounds(sent_of, len(labels))
    out = np.full(len(y), IGNORE, dtype=np.int64)
    keep = first if mode == "first" else last
    out[keep] = y[keep]
    return out

def units(sent_of, n_sents, scope):
    """token ranges encoded independently. document: the whole document (windows give each sentence the
    surrounding sentences as context); sentence: every sentence alone (no context)."""
    if scope == "sentence":
        first, last = sentence_bounds(sent_of, n_sents)
        return list(zip(first.tolist(), (last + 1).tolist()))
    return [(0, len(sent_of))]

# ------------------------------------------------------------------------------------------------ windows
def windows(n, win, stride):
    """(start, end) token ranges covering 0..n; the last window is aligned to the end."""
    if n <= win:
        return [(0, n)]
    out = list(range(0, n - win, stride)) + [n - win]
    return [(s, s + win) for s in sorted(set(out))]

def train_windows(docs, win, stride):
    """docs: {key: (ids, labels, unit ranges)} -> list of (ids_list, labels_list); windows with no label are dropped"""
    out = []
    for ids, y, rngs in docs.values():
        for us, ue in rngs:
            for s, e in windows(ue - us, win, stride):
                yy = y[us + s:us + e]
                if (yy != IGNORE).any():
                    out.append((ids[us + s:us + e], yy.tolist()))
    return out

def batches(wins, batch_size, win, shuffle, rng, max_seqs=256):
    """Batches of EQUAL-length sequences (your models have no padding mask). Short sequences (sentence scope)
    are packed so that a batch holds about batch_size x window tokens."""
    by_len = {}
    for w in wins:
        by_len.setdefault(len(w[0]), []).append(w)
    out = []
    for n, group in by_len.items():
        k = max(1, min(max_seqs, (batch_size * win) // n))
        idx = rng.permutation(len(group)) if shuffle else np.arange(len(group))
        for i in range(0, len(group), k):
            out.append([group[j] for j in idx[i:i + k]])
    if shuffle:
        out = [out[i] for i in rng.permutation(len(out))]
    return out

@torch.no_grad()
def predict_tokens(model, ids, rngs, win, stride, device, batch_size=8, max_seqs=256):
    """Tag scores (n_tokens, n_labels) for one document. Every unit range is cut into windows; each token's scores
    come from the window in which it is most central (largest distance to a window edge inside its unit)."""
    n = len(ids)
    out = np.zeros((n, len(LABELS)), dtype=np.float32)
    best = np.full(n, -1)
    jobs = [(us, ue, us + s, us + e) for us, ue in rngs for s, e in windows(ue - us, win, stride)]
    by_len = {}
    for j in jobs:
        by_len.setdefault(j[3] - j[2], []).append(j)
    x_all = torch.tensor(ids, dtype=torch.long)
    for L_, group in by_len.items():
        k = max(1, min(max_seqs, (batch_size * win) // L_))
        for i in range(0, len(group), k):
            chunk = group[i:i + k]
            x = torch.stack([x_all[s:e] for _, _, s, e in chunk]).to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):
                logits = model(x).float().cpu().numpy()
            for b, (us, ue, s, e) in enumerate(chunk):
                pos = np.arange(s, e)
                left = pos - s if s > us else np.full(e - s, n)
                right = e - 1 - pos if e < ue else np.full(e - s, n)
                score = np.minimum(left, right)
                take = score > best[s:e]
                out[s:e][take] = logits[b][take]
                best[s:e][take] = score[take]
    return out

# ------------------------------------------------------------------------------------------------ sentences
def log_softmax(x):
    x = np.asarray(x, dtype=np.float32)
    m = x.max(-1, keepdims=True)
    return x - m - np.log(np.exp(x - m).sum(-1, keepdims=True))

POOLS = ("mean", "first", "last")

def pool_sentences(token_scores, sent_of, n_sents):
    """token scores -> sentence log-probabilities, three ways:
    mean = average of the tokens' log-probabilities; first / last = the sentence's first / last token."""
    lp = log_softmax(token_scores)
    first, last = sentence_bounds(sent_of, n_sents)
    sums = np.zeros((n_sents, lp.shape[1]), dtype=np.float32)
    np.add.at(sums, sent_of, lp)
    mean = sums / np.maximum(np.bincount(sent_of, minlength=n_sents), 1)[:, None]
    return {"mean": log_softmax(mean), "first": lp[first], "last": lp[last]}

def pools_for(label_tokens):
    """a model trained with labels on the first / last token only has trained scores there"""
    return list(POOLS) if label_tokens == "all" else [label_tokens]

# ------------------------------------------------------------------------------------------------ decoding
def label_statistics(docs):
    """From TRAINING labels only: log prior of each label, log start probabilities, log transition matrix
    (add-one smoothing). Label-sequence statistics, no model."""
    K = len(LABELS)
    prior, start, trans = np.ones(K), np.ones(K), np.ones((K, K))
    for d in docs:
        y = [L2I[l] for l in d["labels"]]
        if not y:
            continue
        start[y[0]] += 1
        for a, b in zip(y, y[1:]):
            trans[a, b] += 1
        for a in y:
            prior[a] += 1
    norm = lambda v: np.log(v / v.sum(-1, keepdims=True)).astype(np.float32)
    return {"log_prior": norm(prior), "log_start": norm(start), "log_trans": norm(trans)}

def decode_doc(logp, stats, lam=0.0, tau=0.0):
    """sentence log-probs (n, 13) -> label ids.
    tau: prior correction — subtract tau x log(prior) (Menon et al. 2021, logit adjustment): rare roles are pushed up,
         which is what macro-F1 rewards. lam: weight of the label-transition scores in a Viterbi over the sentence
         sequence (roles come in blocks); lam = 0 -> each sentence on its own (argmax)."""
    e = logp - tau * stats["log_prior"]
    if lam == 0 or len(e) == 0:
        return e.argmax(-1)
    T, S = lam * stats["log_trans"], lam * stats["log_start"]
    n, K = e.shape
    back = np.zeros((n, K), dtype=np.int64)
    score = e[0] + S
    cols = np.arange(K)
    for t in range(1, n):
        cand = score[:, None] + T
        back[t] = cand.argmax(0)
        score = cand[back[t], cols] + e[t]
    path = np.zeros(n, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

# ------------------------------------------------------------------------------------------------ metrics
def macro_scores(gold, pred, labels=None):
    """Macro P / R / F1 over `labels` (default: all 13) — same rules as evaluate_phase3.py (sklearn, zero_division=0)"""
    labels = labels or LABELS
    st = {l: [0, 0, 0] for l in labels}
    for g, p in zip(gold, pred):
        if g == p:
            if g in st: st[g][0] += 1
        else:
            if p in st: st[p][1] += 1
            if g in st: st[g][2] += 1
    per = {}
    for l, (tp, fp, fn) in st.items():
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = {"P": pr, "R": rc, "F1": 2 * pr * rc / (pr + rc) if pr + rc else 0.0, "support": tp + fn}
    m = lambda k: float(np.mean([per[l][k] for l in labels]))
    return {"macro_f1": m("F1"), "macro_p": m("P"), "macro_r": m("R"), "per_label": per}

def score_docs(docs, preds):
    """official score (both domains pooled, 13 labels) + each domain separately (labels that occur in that domain)"""
    g = [l for d in docs for l in d["labels"]]
    p = [l for d in docs for l in preds[d["key"]]]
    out = macro_scores(g, p)
    for dom in DOMAINS:
        gd = [l for d in docs if d["domain"] == dom for l in d["labels"]]
        pd = [l for d in docs if d["domain"] == dom for l in preds[d["key"]]]
        present = [l for l in LABELS if l in set(gd) | set(pd)]
        out[f"f1_{dom}"] = macro_scores(gd, pd, present)["macro_f1"] if gd else None
    return out

# ------------------------------------------------------------------------------------------------ backbones
TINY = {   # used only by the CPU smoke test (random weights, no checkpoint)
    "decoder":   dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "prefix":    dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "encdec":    dict(d_model=32, n_enc_layers=2, n_dec_layers=1, n_heads=4, max_len=256, max_tgt_len=32, dropout=0.0),
    "swa":       dict(d_model=32, n_layers=2, n_heads=4, window=16, max_len=256, dropout=0.0),
    "diffusion": dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "moe":       dict(d_model=32, n_layers=2, n_heads=4, d_ff=64, n_experts=4, max_len=256, dropout=0.0),
}

class Backbone(nn.Module):
    """Wraps one of YOUR Phase-1 models so that backbone(ids) returns hidden states (B, T, d_model) instead of
    next-token scores (output layer replaced by Identity). encdec: the ENCODER output is used.
    context="echo" (causal models): the window is fed twice and the second copy's states are used, so every
    token has seen the whole window (Springer et al. 2024)."""
    def __init__(self, arch, lm, bos_id, context="plain"):
        super().__init__()
        self.arch, self.lm, self.bos_id, self.context = arch, lm, bos_id, context
        self.kind = getattr(lm, "kind", "causal")
        self.d_model = lm.tok_emb.embedding_dim
        self.max_input = lm.max_len // 2 if context == "echo" else lm.max_len
        lm.head = nn.Identity()
        if arch == "encdec":
            del lm.dec_layers, lm.dec_norm, lm.tgt_pos_emb

    def forward(self, ids):
        T0 = ids.size(1)
        x = torch.cat([ids, ids], 1) if self.context == "echo" else ids
        T = x.size(1)
        if self.arch == "encdec":
            lm = self.lm
            h = lm.tok_emb(x) + lm.src_pos_emb(torch.arange(T, device=x.device))
            for layer in lm.enc_layers:
                h = layer(h)
            h = lm.enc_norm(h)
        elif self.kind == "prefix":
            h = self.lm(x, prefix_len=T)
        else:
            h = self.lm(x)
        return h[:, T - T0:] if self.context == "echo" else h

def build_lm(arch, cfg):
    import models, models_x
    return (models_x if arch in ("diffusion", "moe") else models).build_model(arch, vocab_size=len(bpe_tok_bpe().itos), **cfg)

def load_backbone(arch, ckpt_pattern, context="plain", smoke=False):
    """Rebuilds your Phase-1 model from its checkpoint (its saved cfg + weights) and wraps it."""
    bpe = bpe_tok_bpe()
    if smoke:
        lm = build_lm(arch, TINY[arch])
    else:
        ck = torch.load(find_input(ckpt_pattern), map_location="cpu", weights_only=False)
        assert ck.get("arch", arch) == arch, f"checkpoint {ckpt_pattern} is a '{ck.get('arch')}' model, not '{arch}'"
        lm = build_lm(arch, ck["cfg"])
        lm.load_state_dict(ck["model"])
    return Backbone(arch, lm, bpe.stoi["<s>"], context)

def iter_adaptable(module, skip=("head",)):
    """Every weight a low-rank method can adapt: (name, owner_module, parameter_name, weight) for nn.Linear weights
    and the packed q/k/v weight of nn.MultiheadAttention ('in_proj_weight'). Names containing `skip` are left out."""
    for name, mod in module.named_modules():
        if any(s in name.split(".") for s in skip):
            continue
        if isinstance(mod, nn.MultiheadAttention) and getattr(mod, "in_proj_weight", None) is not None:
            yield name, mod, "in_proj_weight", mod.in_proj_weight
        elif isinstance(mod, nn.Linear):
            yield name, mod, "weight", mod.weight

def layer_stack(backbone):
    """(attribute name, transformer layers, final LayerNorm) of the part the head reads"""
    lm = backbone.lm
    if backbone.arch == "encdec":
        return "enc_layers", lm.enc_layers, lm.enc_norm
    if hasattr(lm, "blocks"):
        return "blocks", lm.blocks, lm.norm
    return "layers", lm.layers, lm.norm

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train

# ------------------------------------------------------------------------------------------------ runs + files
def find_runs(filename, overrides=None):
    """{experiment: folder} for every '.../rr_runs/<experiment>/<filename>' under /kaggle/working and /kaggle/input.
    Found more than once: /kaggle/working wins, then the newest file. overrides = {experiment: folder}."""
    hits = {}
    for rank, root in enumerate((WORK, INPUT_ROOT)):
        for p in glob.glob(f"{root}/**/{filename}", recursive=True):
            folder = os.path.dirname(p)
            if os.path.basename(os.path.dirname(folder)) != "rr_runs":
                continue
            exp, key = os.path.basename(folder), (rank, -os.path.getmtime(p))
            if exp not in hits or key < hits[exp][0]:
                hits[exp] = (key, folder)
    out = {e: f for e, (_, f) in sorted(hits.items())}
    out.update(overrides or {})
    return out

def to_submission(docs, preds):
    """{document id: [label, ...]} in the official format (one label per sentence, original order)"""
    out = {}
    for d in docs:
        p = list(preds[d["key"]])
        assert len(p) == len(d["sents"]), f"{d['id']}: {len(p)} predictions for {len(d['sents'])} sentences"
        out[d["id"]] = p
    return out

def write_submission(docs, preds, path_json, path_zip=None):
    json.dump(to_submission(docs, preds), open(path_json, "w", encoding="utf-8"), ensure_ascii=False)
    if path_zip:
        with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(path_json, arcname="prediction.json")

def write_truth(docs, folder):
    """per-domain truth files in the format evaluate_phase3.py expects (--truth-cl / --truth-it)"""
    os.makedirs(folder, exist_ok=True)
    for dom in DOMAINS:
        json.dump({d["id"]: d["labels"] for d in docs if d["domain"] == dom},
                  open(f"{folder}/dev_truth_{dom}.json", "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(LABELS, open(f"{folder}/labels.json", "w"))

## N5. Your Phase-1 model code (unchanged copies of your `models.py` and `models_x.py`)

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

In [ ]:
%%writefile models_x.py

# =====================================================================================================
# CS-781 Phase 1 — two extra architectures
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
# PART 0 — HELPERS
# =====================================================================================================

def causal_mask(T, device):
    return torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    ).triu(diagonal=1)


def make_layers(n_layers, d_model, n_heads, dropout):
    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(
            module.weight,
            mean=0.0,
            std=0.02
        )

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
# MODEL 5 — LexDiffusionLM
# =====================================================================================================

class LexDiffusionLM(nn.Module):
    kind = "diffusion"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer layers
        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Fully bidirectional attention.
        # NO MASK.
        for layer in self.layers:
            x = layer(x)

        # Final normalization + output
        x = self.norm(x)

        return self.head(x)


# =====================================================================================================
# MODEL 6 — LexMoE
# =====================================================================================================

class MoEFeedForward(nn.Module):
    """
    Top-1 Mixture-of-Experts feed-forward network.

    Each token is routed to exactly one expert.
    """

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        self.n_experts = n_experts

        # Filled during forward()
        self.aux_loss = None
        self.last_choice = None

        # Router
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False
        )

        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):
        B, T, D = x.shape

        # Flatten tokens
        flat = x.reshape(
            B * T,
            D
        )

        # Router probabilities
        router_logits = self.router(flat)

        # Float32 router softmax for numerical stability
        probs = torch.softmax(
            router_logits.float(),
            dim=-1
        )

        # Best expert for each token
        top_p, top_i = probs.max(
            dim=-1
        )

        # Output buffer
        out = flat.new_zeros(
            flat.shape
        )

        # ------------------------------------------------
        # Route tokens to experts
        # ------------------------------------------------

        for e in range(self.n_experts):

            sel = (top_i == e)

            if sel.any():

                expert_out = self.experts[e](
                    flat[sel]
                )

                expert_out = (
                    expert_out
                    * top_p[sel][:, None]
                )

                out[sel] = expert_out.to(
                    out.dtype
                )


        # ------------------------------------------------
        # Switch load-balancing loss
        # ------------------------------------------------

        counts = torch.bincount(
            top_i,
            minlength=self.n_experts
        ).float()

        frac_e = counts / flat.shape[0]

        prob_e = probs.mean(
            dim=0
        )

        self.aux_loss = (
            self.n_experts
            * torch.sum(frac_e * prob_e)
        )


        # Store routing decisions
        self.last_choice = top_i.reshape(
            B,
            T
        )


        # Restore original shape
        return out.reshape(
            B,
            T,
            D
        )


# =====================================================================================================
# MoE BLOCK
# =====================================================================================================

class MoEBlock(nn.Module):
    """
    Pre-norm Transformer block:

        x -> LayerNorm -> Attention -> residual
          -> LayerNorm -> MoE      -> residual
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        # Pre-norm attention
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.drop = nn.Dropout(
            dropout
        )

        # Pre-norm MoE
        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.moe = MoEFeedForward(
            d_model,
            d_ff,
            n_experts,
            dropout
        )


    def forward(self, x, mask):

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + self.drop(
            attn_out
        )


        # ------------------------------------------------
        # MoE feed-forward
        # ------------------------------------------------

        moe_out = self.moe(
            self.norm2(x)
        )

        x = x + moe_out

        return x


# =====================================================================================================
# LexMoE
# =====================================================================================================

class LexMoE(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        n_experts,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Filled during forward()
        self.aux_loss = None

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # MoE Transformer blocks
        self.blocks = nn.ModuleList([
            MoEBlock(
                d_model,
                n_heads,
                d_ff,
                n_experts,
                dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Causal mask
        mask = causal_mask(
            T,
            idx.device
        )

        # Store auxiliary losses
        aux_losses = []

        # Transformer blocks
        for block in self.blocks:

            x = block(
                x,
                mask
            )

            aux_losses.append(
                block.moe.aux_loss
            )

        # Average auxiliary loss across layers
        self.aux_loss = torch.stack(
            aux_losses
        ).mean()

        # Final normalization
        x = self.norm(x)

        # Vocabulary logits
        return self.head(x)


# =====================================================================================================
# FACTORY — DO NOT CHANGE
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "diffusion": LexDiffusionLM,
        "moe": LexMoE
    }

    return classes[arch](
        vocab_size,
        **cfg
    )

## N6. `rr_methods.py` — YOUR CODE
Your Phase-2 `ner_methods.py` unchanged (only its import now reads `rr_common`). It works as-is for RR: the head
gets 13 outputs. Section 6 at the end has stubs for **rsLoRA** and **IA³** — §9.2.1 needs one more listed technique
besides LoRA, LoRMA and distillation; write one and add an experiment with `"method": "rslora"` or `"ia3"`.

In [ ]:
%%writefile rr_methods.py
# =====================================================================================================
#  CS-781 Phase 3 (copied from Phase 2) — YOUR CODE: the NER head, the fine-tuning methods, and knowledge distillation
#                                                                AUTHOR: <your name>   (your own code)
# =====================================================================================================

import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
import rr_common as L


# =====================================================================================================
#  1. THE NER HEAD
# =====================================================================================================

class TokenClassifier(nn.Module):

    def __init__(self, backbone, n_tags, dropout):
        super().__init__()

        # Keep backbone
        self.backbone = backbone

        # Dropout before classifier
        self.dropout = nn.Dropout(dropout)

        # Token-level classification head
        self.head = nn.Linear(
            backbone.d_model,
            n_tags
        )


    def forward(self, ids):

        # Backbone hidden states
        h = self.backbone(ids)

        # Classification
        return self.head(
            self.dropout(h)
        )


# =====================================================================================================
#  2. SIMPLE METHODS: full, partial, BitFit
# =====================================================================================================

def method_full(model, **cfg):
    """Full fine-tuning: every parameter of the backbone trains."""

    for p in model.backbone.parameters():
        p.requires_grad_(True)


def method_partial(model, n_last=2, **cfg):
    """
    Partial / layer-wise fine-tuning:
    only the LAST n_last transformer layers + final LayerNorm train.
    """

    # Get the relevant layer stack
    name, layers, final_norm = L.layer_stack(
        model.backbone
    )

    # Train only the last n_last layers
    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad_(True)

    # Train final LayerNorm
    for p in final_norm.parameters():
        p.requires_grad_(True)


def method_bitfit(model, **cfg):
    """
    BitFit:
    only bias parameters of the backbone train.
    """

    for name, p in model.backbone.named_parameters():

        if name.endswith("bias"):
            p.requires_grad_(True)


# =====================================================================================================
#  3. LoRA
# =====================================================================================================

class LoRAParam(nn.Module):
    """
    Additive low-rank update:

        W' = W + (alpha / r) * B @ A
    """

    def __init__(
        self,
        d_out,
        d_in,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.empty(r, d_in)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5)
        )

        # B: zero initialization
        # This guarantees W' = W initially.
        self.B = nn.Parameter(
            torch.zeros(d_out, r)
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        return W + self.scale * (
            self.B @ self.A
        )


def method_lora(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRAParam(
                d_out,
                d_in,
                r,
                alpha
            )
        )


# =====================================================================================================
#  4. LoRMA
# =====================================================================================================

class LoRMAParam(nn.Module):
    """
    LoRMA+ multiplicative low-rank update:

        W' = W + (alpha / r) * B @ (A @ W)

    where:

        A : (r, d_out)
        B : (d_out, r)
    """

    def __init__(
        self,
        d_out,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.randn(
                r,
                d_out
            ) * 0.02
        )

        # B: zero initialization
        self.B = nn.Parameter(
            torch.zeros(
                d_out,
                r
            )
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        # First compute A @ W
        AW = self.A @ W

        # Then B @ (A @ W)
        update = self.B @ AW

        return W + self.scale * update


def method_lorma(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRMAParam(
                d_out,
                r,
                alpha
            )
        )


# =====================================================================================================
#  5. KNOWLEDGE DISTILLATION
# =====================================================================================================

def kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):
    """
    Combined hard-label cross entropy and
    teacher-student KL divergence.
    """

    # Only real labelled tokens
    mask = labels != -100

    s = student_logits[mask]
    t = teacher_logits[mask]
    y = labels[mask]

    # --------------------------------------------------------
    # Hard-label loss
    # --------------------------------------------------------

    hard = F.cross_entropy(
        s,
        y
    )

    # --------------------------------------------------------
    # Soft teacher/student distributions
    # --------------------------------------------------------

    soft = F.kl_div(
        F.log_softmax(
            s / temperature,
            dim=-1
        ),
        F.softmax(
            t / temperature,
            dim=-1
        ),
        reduction="batchmean"
    )

    # Hinton temperature scaling
    soft = soft * (
        temperature * temperature
    )

    # Combined loss
    return (
        alpha * hard
        + (1.0 - alpha) * soft
    )


def make_student(
    backbone,
    keep_layers
):
    """
    Make a smaller copy of a Phase-1 backbone,
    keeping only its first keep_layers layers.
    """

    # Deep copy so original backbone is untouched
    small = copy.deepcopy(
        backbone
    )

    # Find the layer stack
    name, layers, _ = L.layer_stack(
        small
    )

    # Keep only first keep_layers
    setattr(
        small.lm,
        name,
        nn.ModuleList(
            list(layers)[:keep_layers]
        )
    )

    return small


# =====================================================================================================
#  METHOD TABLE
# =====================================================================================================

METHODS = {
    "full": method_full,
    "partial": method_partial,
    "bitfit": method_bitfit,
    "lora": method_lora,
    "lorma": method_lorma,
}


# =====================================================================================================
#  6. NEW FOR PHASE 3 — WRITE THESE TWO FUNCTIONS YOURSELF (course rule §17.5: method code must be your own)
# =====================================================================================================
#
#  WHY YOU NEED THEM
#  The Phase-3 brief (§9.2.1) only accepts techniques from its list: LoRA, QLoRA, AdaLoRA, rsLoRA, LORMA,
#  Prefix Tuning, Prompt Tuning, IA³, Knowledge Distillation. You already have LoRA, LoRMA and distillation.
#  Full fine-tuning and BitFit are not on the list. rsLoRA and IA³ below give you two more listed techniques.
#  N3 already contains one experiment for each ("decoder_rslora", "decoder_ia3"), so as soon as you write them,
#  Save & Run All trains them and the lab makes their zips and puts them on the score chart.
#  If you leave one unwritten, only that experiment stops (with NotImplementedError); everything else runs.
#
#  HOW TO CHECK YOUR CODE BEFORE THE GPU RUN
#  Run this cell, then N9 (the CPU smoke test). Both "decoder_rslora" and "decoder_ia3" must print "exit code 0".
#
# -----------------------------------------------------------------------------------------------------
#  A. rsLoRA — "rank-stabilised LoRA" (Kalajdzievski 2023)
# -----------------------------------------------------------------------------------------------------
#  In plain English:
#    rsLoRA is your LoRA from section 3 with one number changed. LoRA multiplies its update B @ A by
#    alpha divided by r. rsLoRA multiplies it by alpha divided by the SQUARE ROOT of r. Nothing else changes:
#    same A and B matrices, same starting values, same weights adapted.
#
#  What your function has to do, in order:
#    1. Go through every weight that can be adapted, exactly the way your method_lora does it
#       (the same "for name, owner, pname, W in L.iter_adaptable(...)" loop).
#    2. Inside the loop, read the weight's shape into d_out and d_in, as method_lora does.
#    3. Create one LoRAParam object for that weight with d_out, d_in, r and alpha — the class you already wrote.
#       Keep it in a variable instead of passing it straight into the register call.
#    4. That object computed its scale as alpha / r when it was created. Overwrite its "scale" attribute with
#       alpha divided by the square root of r (math.sqrt is already imported at the top of this cell).
#    5. Attach the object to the weight with parametrize.register_parametrization(owner, pname, your_object),
#       the same call method_lora uses.
#    6. Delete the "raise NotImplementedError" line.
#  Tip: copy the body of method_lora, then change steps 3 and 4. It is about six lines.

def method_rslora(model, r=8, alpha=16, skip=("head",), **cfg):
    """Rank-stabilised LoRA: use alpha / sqrt(r) as the update scale."""
    if r <= 0:
        raise ValueError("r must be a positive integer")

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out, d_in = W.shape
        adapter = LoRAParam(d_out, d_in, r, alpha)
        adapter.scale = alpha / math.sqrt(r)
        parametrize.register_parametrization(owner, pname, adapter)


# -----------------------------------------------------------------------------------------------------
#  B. IA³ — "Infused Adapter by Inhibiting and Amplifying Inner Activations" (Liu et al. 2022)
# -----------------------------------------------------------------------------------------------------
#  In plain English:
#    IA³ adds no matrices at all. Every adapted weight gets ONE learned vector, with one number per output
#    row of the weight. Each row of the weight is multiplied by its number. The vector starts as all ones, so
#    at the start the model is unchanged (the same idea as B starting at zero in LoRA). Only these vectors and
#    the head are trained, so IA³ trains far fewer parameters than LoRA.
#
#  What you have to write — two pieces:
#
#    Piece 1 — a small class, written just above method_ia3, shaped like your LoRAParam:
#      1. It inherits from nn.Module and its __init__ receives d_out (the number of output rows of the weight).
#      2. In __init__, after super().__init__(), create one trainable vector of length d_out filled with ones
#         (an nn.Parameter wrapped around torch.ones of size d_out). Call it something like "l".
#      3. Its forward receives the original weight W (shape d_out x d_in) and returns W with every row multiplied
#         by the matching number of the vector. To do that, turn the vector into a column of shape d_out x 1
#         (index it with [:, None]) and multiply it with W — PyTorch repeats the column across all d_in columns.
#
#    Piece 2 — the function method_ia3:
#      4. Go through every adaptable weight with the same loop as method_lora.
#      5. Read the weight's number of output rows (the first number of W.shape).
#      6. Create your Piece-1 object with that number and attach it with parametrize.register_parametrization,
#         exactly like method_lora attaches LoRAParam.
#      7. Delete the "raise NotImplementedError" line.
#
#    For your report: the paper rescales only keys, values and the feed-forward hidden layer. Rescaling every
#    adaptable weight (what steps 4-6 do) is a simpler variant — say which one you used.

class IA3Param(nn.Module):
    """Row-wise IA³ scaling for a weight matrix."""

    def __init__(self, d_out):
        super().__init__()
        self.l = nn.Parameter(torch.ones(d_out))

    def forward(self, W):
        return self.l[:, None] * W


def method_ia3(model, skip=("head",), **cfg):
    """Apply trainable row-wise scaling to every adaptable backbone weight."""
    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out = W.shape[0]
        parametrize.register_parametrization(
            owner,
            pname,
            IA3Param(d_out),
        )


# Do not change this line: it makes "rslora" and "ia3" usable as "method" in N3.
METHODS.update({"rslora": method_rslora, "ia3": method_ia3})

## N7. Scripts: training (one experiment), the lab, the official Phase-3 evaluator

In [ ]:
%%writefile train_rr.py
"""Runs ONE experiment from run_config_rr.json: builds your Phase-1 backbone + your head (TokenClassifier with
13 outputs), applies your fine-tuning method, trains on the CL + IT training documents, keeps the epoch with the
best macro-F1 on the official CL + IT dev documents, then writes dev + test predictions, the Codabench zip, and the
sentence log-probabilities of every pooling (dev_probs.pkl, test_probs.pkl) for the lab (N11).
   python train_rr.py --exp NAME --device cuda:0        (real run)
   python train_rr.py --exp NAME --device cpu --smoke   (tiny random model, few documents: checks the code)"""
import argparse, json, os, pickle, random, time
import numpy as np, torch, torch.nn.functional as F
import rr_common as L
import rr_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
cuda = dev.startswith("cuda")
if cuda:
    torch.cuda.set_device(dev)

C = json.load(open(f"{L.WORK}/run_config_rr.json"))
def exp_cfg(name):
    if name in C["experiments"]:
        return {**C["defaults"], **C["experiments"][name], "name": name}
    raise KeyError(f"experiment '{name}' is not in EXPERIMENTS (cell N3)")
E = exp_cfg(args.exp)
RUNS = f"{L.WORK}/{'rr_runs_smoke' if args.smoke else 'rr_runs'}"

def best_epoch_of(name):
    p = f"{RUNS}/{name}/summary.json"
    if not os.path.exists(p):
        p = f"{L.find_runs('summary.json')[name]}/summary.json"
    return int(json.load(open(p))["best_epoch"])

if E["train_on_all"] and E.get("epochs_from") and not args.smoke:
    E["epochs"] = best_epoch_of(E["epochs_from"])
if args.smoke:
    E.update(window=96, stride=64, epochs=1, batch_size=4, patience=99)
OUT = f"{RUNS}/{args.exp}"; os.makedirs(OUT, exist_ok=True)
log = open(f"{OUT}/log.jsonl", "w")
t0 = time.time()
def say(**kw):
    kw["t"] = round(time.time() - t0); log.write(json.dumps(kw) + "\n"); log.flush()
    print(args.exp, json.dumps(kw), flush=True)
seed = E.get("seed", C["seed"]); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------------------------------------------ data
train, dev_docs, test = L.load_data()
if E["train_on_all"]:
    train = train + dev_docs                    # official dev documents join training; no model selection
if args.smoke:
    pick = lambda docs, k: [d for dom in L.DOMAINS for d in [x for x in docs if x["domain"] == dom][:k]]
    train, dev_docs, test = pick(train, 2), pick(dev_docs, 1), pick(test, 1)
tok = L.SentTokenizer()
enc = {d["key"]: tok.doc(d["sents"]) for d in train + dev_docs + test}
rngs = {k: L.units(so, int(so.max()) + 1, E["scope"]) for k, (ids, so) in enc.items()}
train_docs = {d["key"]: (enc[d["key"]][0], L.token_labels(enc[d["key"]][1], d["labels"], E["label_tokens"]),
                         rngs[d["key"]]) for d in train}

# ------------------------------------------------------------------ model
def build(e, smoke):
    backbone = L.load_backbone(e["arch"], C["checkpoints"][e["arch"]], context=e["context"], smoke=smoke)
    if e.get("student_layers"):
        backbone = M.make_student(backbone, e["student_layers"])
    model = M.TokenClassifier(backbone, n_tags=len(L.LABELS), dropout=e["head_dropout"])
    for p in model.parameters():
        p.requires_grad_(False)
    M.METHODS[e["method"]](model, **e["method_cfg"])
    for p in model.head.parameters():          # the new head is always trained
        p.requires_grad_(True)
    return model

model = build(E, args.smoke).to(dev)
def fit_window(max_input):
    w = min(E["window"], max_input)
    return w, max(1, min(E["stride"], round(w * E["stride"] / E["window"])))
win, stride = fit_window(model.backbone.max_input)
total, trainable = L.count_params(model)
say(event="start", arch=E["arch"], method=E["method"], params_total=total, params_trainable=trainable,
    trainable_pct=round(100 * trainable / total, 3), window=win, stride=stride, epochs=E["epochs"], seed=seed,
    scope=E["scope"], label_tokens=E["label_tokens"], pool=E["pool"], train_docs=len(train), dev_docs=len(dev_docs))

teacher = None
if E.get("teacher"):
    TE = exp_cfg(E["teacher"])
    tpath = f"{RUNS}/{E['teacher']}/best.pt"
    if not os.path.exists(tpath):
        tpath = f"{L.find_runs('best.pt')[E['teacher']]}/best.pt"      # teacher trained in an earlier session
    teacher = build(TE, args.smoke)
    teacher.load_state_dict(torch.load(tpath, map_location="cpu", weights_only=False)["state"])
    teacher.to(dev).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    win, stride = fit_window(min(win, teacher.backbone.max_input))
    say(event="teacher", name=E["teacher"], params_total=L.count_params(teacher)[0])

# ------------------------------------------------------------------ training setup
wins = L.train_windows(train_docs, win, stride)
lab_counts = np.bincount([t for _, y in wins for t in y if t != L.IGNORE], minlength=len(L.LABELS))
weights = None
if E["class_weights"] == "inv_sqrt":
    w = 1.0 / np.sqrt(np.maximum(lab_counts, 1)); w = w / w.mean()
    weights = torch.tensor(w, dtype=torch.float32, device=dev)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=E["lr"], weight_decay=E["weight_decay"])
rng = np.random.RandomState(seed)
steps_per_epoch = len(L.batches(wins, E["batch_size"], win, False, rng))
if args.smoke:
    steps_per_epoch = min(steps_per_epoch, 6)
total_steps = steps_per_epoch * E["epochs"]
warm = max(1, int(E["warmup_frac"] * total_steps))
def lr_at(step):
    return E["lr"] * min((step + 1) / warm, max(0.0, (total_steps - step) / max(1, total_steps - warm)))
scaler = torch.amp.GradScaler("cuda", enabled=cuda)

def sentence_probs(docs):
    """{pool: {doc key: (n_sentences, 13) log-probs}} for the pools this run supports"""
    model.eval()
    out = {p: {} for p in L.pools_for(E["label_tokens"])}
    for d in docs:
        ids, so = enc[d["key"]]
        sc = L.predict_tokens(model, ids, rngs[d["key"]], win, stride, dev, E["batch_size"])
        pooled = L.pool_sentences(sc, so, len(d["sents"]))
        for p in out:
            out[p][d["key"]] = pooled[p].astype(np.float16)
    model.train()
    return out

def predict(docs, probs):
    return {d["key"]: [L.LABELS[i] for i in probs[E["pool"]][d["key"]].astype(np.float32).argmax(-1)] for d in docs}

# ------------------------------------------------------------------ train
best, best_epoch, bad, step = -1.0, -1, 0, 0
model.train()
if cuda:
    torch.cuda.reset_peak_memory_stats()
for epoch in range(1, E["epochs"] + 1):
    te, losses = time.time(), []
    for bi, batch in enumerate(L.batches(wins, E["batch_size"], win, True, rng)):
        if args.smoke and bi >= steps_per_epoch:
            break
        x = torch.tensor([w[0] for w in batch], dtype=torch.long, device=dev)
        y = torch.tensor([w[1] for w in batch], dtype=torch.long, device=dev)
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
            logits = model(x).float()
            if teacher is not None:
                with torch.no_grad():
                    t_logits = teacher(x).float()
                loss = M.kd_loss(logits, t_logits, y, **E["kd"])
            else:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1),
                                       weight=weights, ignore_index=L.IGNORE)
            aux = getattr(model.backbone.lm, "aux_loss", None)
            if E["moe_aux_coef"] and aux is not None:
                loss = loss + E["moe_aux_coef"] * aux
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, E["grad_clip"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        losses.append(loss.item()); step += 1
    epoch_sec = time.time() - te
    if E["train_on_all"]:
        say(event="epoch", epoch=epoch, train_loss=round(float(np.mean(losses)), 4), sec=round(epoch_sec))
        torch.save({"state": model.state_dict(), "exp": E, "epoch": epoch}, f"{OUT}/best.pt"); best_epoch = epoch
        continue
    s = L.score_docs(dev_docs, predict(dev_docs, sentence_probs(dev_docs)))
    say(event="epoch", epoch=epoch, train_loss=round(float(np.mean(losses)), 4), dev_macro_f1=round(s["macro_f1"], 4),
        dev_macro_p=round(s["macro_p"], 4), dev_macro_r=round(s["macro_r"], 4),
        dev_f1_CL=s["f1_CL"] and round(s["f1_CL"], 4), dev_f1_IT=s["f1_IT"] and round(s["f1_IT"], 4), sec=round(epoch_sec))
    if s["macro_f1"] > best:
        best, best_epoch, bad = s["macro_f1"], epoch, 0
        torch.save({"state": model.state_dict(), "exp": E, "epoch": epoch}, f"{OUT}/best.pt")
    else:
        bad += 1
        if bad >= E["patience"]:
            say(event="early_stop", epoch=epoch)
            break

# ------------------------------------------------------------------ final predictions with the best epoch
model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=dev, weights_only=False)["state"])
peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else None
summary = {"exp": args.exp, "arch": E["arch"], "method": E["method"], "scope": E["scope"], "context": E["context"],
           "label_tokens": E["label_tokens"], "pool": E["pool"], "seed": seed, "train_on_all": E["train_on_all"],
           "epochs": E["epochs"], "best_epoch": best_epoch, "params_total": total, "params_trainable": trainable,
           "peak_gpu_gb": peak, "train_windows": len(wins), "window": win, "stride": stride}
if not E["train_on_all"]:
    dprobs = sentence_probs(dev_docs)
    pickle.dump(dprobs, open(f"{OUT}/dev_probs.pkl", "wb"))
    dpred = predict(dev_docs, dprobs)
    L.write_submission(dev_docs, dpred, f"{OUT}/dev_prediction.json")
    s = L.score_docs(dev_docs, dpred)
    summary.update(dev_macro_f1=round(s["macro_f1"], 4), dev_macro_p=round(s["macro_p"], 4), dev_macro_r=round(s["macro_r"], 4),
                   dev_f1_CL=s["f1_CL"], dev_f1_IT=s["f1_IT"],
                   dev_per_label_f1={l: round(v["F1"], 4) for l, v in s["per_label"].items()})
ti = time.time()
tprobs = sentence_probs(test)
pickle.dump(tprobs, open(f"{OUT}/test_probs.pkl", "wb"))
summary["test_predict_sec"] = round(time.time() - ti, 1)
L.write_submission(test, predict(test, tprobs), f"{OUT}/prediction.json", f"{OUT}/submission_{args.exp}.zip")
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
say(event="done", **{k: v for k, v in summary.items() if k != "dev_per_label_f1"})

In [ ]:
%%writefile rr_lab.py
"""Phase 3 lab — no training. Takes the sentence log-probabilities every finished run saved (dev_probs.pkl,
test_probs.pkl), builds prediction sets from them (pooling x prior correction tau x transition weight lambda,
probability ensembles, greedy ensembles), scores each on the official CL + IT dev documents, writes the test
predictions as Codabench zips, submissions/index.csv, and a score chart. You choose which zips to upload."""
import csv, glob, hashlib, json, os, pickle, time
import numpy as np
import rr_common as L

KIND_COLORS = {"single": "#2a78d6", "ensemble": "#eb6834", "greedy": "#1baf7a"}   # fixed categorical order


class Lab:
    def __init__(self, out_dir=f"{L.WORK}/submissions"):
        self.train, self.dev, self.test = L.load_data()
        self.stats = L.label_statistics(self.train)          # label prior / transitions from TRAINING labels only
        self.out_dir = out_dir
        os.makedirs(f"{out_dir}/dev_predictions", exist_ok=True)
        L.write_truth(self.dev, f"{out_dir}/dev_predictions")
        self.rows, self.models = [], {}

    def load_runs(self, overrides=None):
        for e, f in L.find_runs("test_probs.pkl", overrides).items():
            s = json.load(open(f"{f}/summary.json")) if os.path.exists(f"{f}/summary.json") else {}
            dv = f"{f}/dev_probs.pkl"
            self.models[e] = {"test": pickle.load(open(f"{f}/test_probs.pkl", "rb")),
                              "dev": pickle.load(open(dv, "rb")) if os.path.exists(dv) and not s.get("train_on_all") else None,
                              "folder": f}
            self.models[e]["pools"] = list(self.models[e]["test"])
        print(f"runs found: {len(self.models)}")
        for e, m in self.models.items():
            print(f"  {e:28s} pools {m['pools']} {'' if m['dev'] is not None else '(no dev scores: train_on_all)'} <- {m['folder']}")

    # ------------------------------------------------------------------ prediction sets
    def decode(self, members, split, pool, lam, tau):
        docs = self.dev if split == "dev" else self.test
        if split == "dev" and any(self.models[m]["dev"] is None for m in members):
            return None
        out = {}
        for d in docs:
            lps = [self.models[m][split][pool][d["key"]].astype(np.float32) for m in members]
            lp = lps[0] if len(lps) == 1 else np.log(sum(np.exp(x) for x in lps) / len(lps) + 1e-9)
            out[d["key"]] = [L.LABELS[i] for i in L.decode_doc(lp, self.stats, lam, tau)]
        return out

    def add(self, name, kind, dev_preds, test_preds, **info):
        s = L.score_docs(self.dev, dev_preds) if dev_preds is not None else None
        r = lambda v: None if v is None else round(v, 4)
        row = {"name": name, "kind": kind, "dev_f1": r(s and s["macro_f1"]), "dev_p": r(s and s["macro_p"]),
               "dev_r": r(s and s["macro_r"]), "dev_f1_CL": r(s and s["f1_CL"]), "dev_f1_IT": r(s and s["f1_IT"]), **info}
        self.rows.append((row, dev_preds, test_preds))
        return row["dev_f1"]

    def pools_of(self, members):
        return [p for p in L.POOLS if all(p in self.models[m]["pools"] for m in members)]

    def variants(self, name, kind, members, lams, taus):
        for pool in self.pools_of(members):
            for lam in lams:
                for tau in taus:
                    v, t = self.decode(members, "dev", pool, lam, tau), self.decode(members, "test", pool, lam, tau)
                    self.add(f"{name}__{pool}__lam{lam}__tau{tau}", kind, v, t,
                             members="+".join(members), pool=pool, lam=lam, tau=tau)

    def singles(self, lams, taus):
        for e, m in self.models.items():
            if m["dev"] is None:
                print(f"{e}: no dev scores (train_on_all) -> its zip is the one train_rr.py wrote")
                continue
            self.variants(e, "single", [e], lams, taus)

    def ensembles(self, groups, lams, taus):
        """{name: [members]} -> every pool x lambda x tau variant, scored on dev;
        {name: {"members": [...], "pool": ..., "lam": ..., "tau": ...}} -> that one variant (for train_on_all runs)"""
        for name, g in groups.items():
            members = g["members"] if isinstance(g, dict) else list(g)
            missing = [m for m in members if m not in self.models]
            if missing:
                print(f"ensemble {name}: skipped, no scores for {missing}")
                continue
            if not isinstance(g, dict):
                self.variants(f"ens_{name}", "ensemble", members, lams, taus)
                continue
            v, t = (self.decode(members, sp, g["pool"], g["lam"], g["tau"]) for sp in ("dev", "test"))
            self.add(f"ens_{name}__{g['pool']}__lam{g['lam']}__tau{g['tau']}", "ensemble", v, t,
                     members="+".join(members), pool=g["pool"], lam=g["lam"], tau=g["tau"])

    def greedy(self, lams, taus):
        """Greedy forward ensemble selection on dev (Caruana et al. 2004), one per pooling: repeatedly add the run
        that raises dev macro-F1 the most (best lambda x tau of the grid), stop when nothing helps."""
        for pool in L.POOLS:
            pool_runs = [e for e, m in self.models.items() if m["dev"] is not None and pool in m["pools"]]
            def objective(members):
                best = (-1.0, None, None)
                for lam in lams:
                    for tau in taus:
                        f = L.score_docs(self.dev, self.decode(members, "dev", pool, lam, tau))["macro_f1"]
                        if f > best[0]:
                            best = (f, lam, tau)
                return best
            chosen, best = [], (-1.0, None, None)
            while True:
                tries = [(objective(chosen + [e]), e) for e in pool_runs if e not in chosen]
                if not tries:
                    break
                res, e = max(tries, key=lambda x: x[0][0])
                if res[0] <= best[0]:
                    break
                chosen.append(e); best = res
            if len(chosen) < 2:
                print(f"greedy {pool}: no ensemble beats the best single run ({chosen})")
                continue
            f, lam, tau = best
            self.add(f"greedy__{pool}__lam{lam}__tau{tau}", "greedy",
                     self.decode(chosen, "dev", pool, lam, tau), self.decode(chosen, "test", pool, lam, tau),
                     members="+".join(chosen), pool=pool, lam=lam, tau=tau)
            print(f"greedy {pool}: {chosen} -> dev macro-F1 {f:.4f} (lambda {lam}, tau {tau})")

    # ------------------------------------------------------------------ output
    def finish(self, top_n=30):
        """zips for: the top_n sets by dev macro-F1, the best set of each member list, every set without dev scores.
        Identical test predictions are written once. index.csv lists every set."""
        scored = sorted([r for r in self.rows if r[0]["dev_f1"] is not None], key=lambda r: -r[0]["dev_f1"])
        keep = {id(r) for r in scored[:top_n]}
        best_of = {}
        for r in scored:
            best_of.setdefault(r[0].get("members"), r)
        keep |= {id(r) for r in best_of.values()} | {id(r) for r in self.rows if r[0]["dev_f1"] is None}
        order = scored + [r for r in self.rows if r[0]["dev_f1"] is None]
        for f in glob.glob(f"{self.out_dir}/*.zip") + glob.glob(f"{self.out_dir}/dev_predictions/*__*.json"):
            os.remove(f)                                         # files of an earlier finish()
        seen, n = {}, 0
        for r in order:
            row, v, t = r
            row["file"] = ""
            if id(r) not in keep:
                continue
            sub = L.to_submission(self.test, t)
            h = hashlib.md5(json.dumps(sub, sort_keys=True).encode()).hexdigest()
            if h in seen:
                row["file"] = f"(same as {seen[h]})"
                continue
            seen[h] = row["file"] = f"{row['name']}.zip"
            L.write_submission(self.test, t, f"{self.out_dir}/_prediction.json", f"{self.out_dir}/{row['file']}")
            if v is not None:
                json.dump(L.to_submission(self.dev, v), open(f"{self.out_dir}/dev_predictions/{row['name']}.json", "w"))
            n += 1
        if os.path.exists(f"{self.out_dir}/_prediction.json"):
            os.remove(f"{self.out_dir}/_prediction.json")
        cols = ["name", "kind", "dev_f1", "dev_p", "dev_r", "dev_f1_CL", "dev_f1_IT", "members", "pool", "lam", "tau", "file"]
        with open(f"{self.out_dir}/index.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
            w.writeheader()
            for row, _, _ in order:
                w.writerow(row)
        print(f"{len(self.rows)} prediction sets scored, {n} zips written to {self.out_dir}/ (index.csv lists all)")
        return [r[0] for r in order]


def table(rows, n=25):
    scored = [r for r in rows if r["dev_f1"] is not None]
    print(f"{'dev F1':>7s} {'P':>6s} {'R':>6s} {'CL':>6s} {'IT':>6s}  {'prediction set':62s} zip")
    for r in scored[:n]:
        cl, it = r["dev_f1_CL"], r["dev_f1_IT"]
        print(f"{r['dev_f1']:7.4f} {r['dev_p']:6.3f} {r['dev_r']:6.3f} {cl if cl is not None else float('nan'):6.3f} "
              f"{it if it is not None else float('nan'):6.3f}  {r['name'][:62]:62s} {r.get('file', '')}")
    for r in rows:
        if r["dev_f1"] is None and r.get("file"):
            print(f"{'n/a':>7s} {'':6s} {'':6s} {'':6s} {'':6s}  {r['name'][:62]:62s} {r['file']}")


def chart(rows, path, n=25):
    """Tentative score chart (dev macro-F1, the official metric on the official dev documents):
    left = the n best prediction sets, coloured by kind; right = the best set of every run and ensemble, with its
    CL-only and IT-only scores. Saved as PNG; index.csv is the table view of the same numbers."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    scored = [r for r in rows if r["dev_f1"] is not None]
    if not scored:
        print("no dev scores to plot"); return
    top = scored[:n][::-1]
    best_of = {}
    for r in scored:
        best_of.setdefault(r["members"], r)
    groups = list(best_of.values())[:15][::-1]
    ink, muted, grid = "#0b0b0b", "#52514e", "#e4e3df"
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(17, max(6, 0.32 * max(len(top), 3 * len(groups) / 1.6))),
                                 gridspec_kw={"width_ratios": [1.15, 1]})
    fig.patch.set_facecolor("#fcfcfb")
    for ax in (a1, a2):
        ax.set_facecolor("#fcfcfb")
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.spines["bottom"].set_color(grid)
        ax.tick_params(colors=muted, labelsize=8, length=0)
        ax.grid(axis="x", color=grid, linewidth=0.8)
        ax.set_axisbelow(True)
    y = np.arange(len(top))
    a1.barh(y, [r["dev_f1"] for r in top], height=0.62, color=[KIND_COLORS[r["kind"]] for r in top],
            edgecolor="#fcfcfb", linewidth=2)
    a1.set_yticks(y, [r["name"][:58] for r in top], fontsize=7.5, color=ink)
    for yi, r in zip(y, top):
        a1.text(r["dev_f1"] + 0.003, yi, f"{r['dev_f1']:.3f}", va="center", fontsize=7.5, color=muted)
    a1.set_xlim(0, min(1.0, max(r["dev_f1"] for r in top) * 1.12))
    a1.set_xlabel("dev macro-F1 (CL + IT pooled, 13 labels)", color=muted, fontsize=9)
    a1.set_title(f"Top {len(top)} prediction sets", loc="left", color=ink, fontsize=11)
    names = {"single": "single run", "ensemble": "your ensemble", "greedy": "greedy ensemble"}
    kinds = [k for k in KIND_COLORS if any(r["kind"] == k for r in top)]
    a1.legend([plt.Rectangle((0, 0), 1, 1, color=KIND_COLORS[k]) for k in kinds], [names[k] for k in kinds],
              loc="lower right", frameon=False, fontsize=8)
    # right: best set per run / ensemble, overall + per domain
    series = [("dev_f1", "both domains", "#2a78d6"), ("dev_f1_CL", "CL only", "#eb6834"), ("dev_f1_IT", "IT only", "#1baf7a")]
    yy = np.arange(len(groups)); h = 0.26
    for k, (key, lab, col) in enumerate(series):
        vals = [r[key] or 0 for r in groups]
        a2.barh(yy + (1 - k) * h, vals, height=h, color=col, label=lab, edgecolor="#fcfcfb", linewidth=1.5)
    for yi, r in zip(yy, groups):
        a2.text(r["dev_f1"] + 0.003, yi + h, f"{r['dev_f1']:.3f}", va="center", fontsize=7.5, color=muted)
    a2.set_yticks(yy, [r["members"][:48] for r in groups], fontsize=7.5, color=ink)
    a2.set_xlim(0, min(1.0, max(max(r[k] or 0 for r in groups) for k, _, _ in series) * 1.12))
    a2.set_xlabel("dev macro-F1 of the best set of each run / ensemble", color=muted, fontsize=9)
    a2.set_title("Best per run, by domain", loc="left", color=ink, fontsize=11)
    a2.legend(loc="lower right", frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130, facecolor=fig.get_facecolor())
    plt.close(fig)
    print("chart saved:", path)

In [ ]:
%%writefile evaluate_phase3.py
#!/usr/bin/env python3
"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.

Usage:
  python evaluate_phase3.py \
      --truth-cl cl_test_truth.json \
      --truth-it it_test_truth.json \
      --prediction prediction.json \
      --labels labels.json

Each truth file must be a JSON object mapping document IDs to ordered lists of
sentence labels. prediction.json must have the same structure and contain
predictions for documents from both reference files.

labels.json may be either:
  ["Label1", "Label2", ...]
or a label->id / id->label mapping.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            # label -> integer ID
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            # integer/string ID -> label
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument(
        "--truth-cl",
        required=True,
        help="Local held-out reference JSON for the CL split.",
    )
    parser.add_argument(
        "--truth-it",
        required=True,
        help="Local held-out reference JSON for the IT split.",
    )
    parser.add_argument(
        "--prediction",
        required=True,
        help="Participant prediction JSON covering documents from both splits.",
    )
    parser.add_argument(
        "--labels",
        required=True,
        help="Public Phase 3 label vocabulary JSON.",
    )
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [
            f"missing={len(missing)}",
            f"extra={len(extra)}",
        ]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"EVALUATION FAILED: {exc}", file=sys.stderr)
        sys.exit(1)

## N8. Data check (sizes, labels per domain, document lengths, checkpoints)

In [ ]:
# N8 — data check: how the files were read, sizes, label distribution per domain, document lengths in tokens.
# If this cell fails, the data format differs from the brief: paste the error to fix the reader (rr_common._docs).
import json, importlib, numpy as np, torch
from collections import Counter
import rr_common as L
importlib.reload(L)
train, dev, test = L.load_data()
print("labels (official order):", L.LABELS)
for name, docs in (("train", train), ("dev", dev), ("test", test)):
    for dom in L.DOMAINS:
        ds = [d for d in docs if d["domain"] == dom]
        print(f"{name:5s} {dom}: {len(ds):4d} documents, {sum(len(d['sents']) for d in ds):6d} sentences")
ids = [d["id"] for d in test]
assert len(ids) == len(set(ids)), "test document ids repeat across CL and IT"
print("\nexample test document:", test[0]["id"], "|", len(test[0]["sents"]), "sentences | first:", test[0]["sents"][0][:120])
tab = {dom: Counter(l for d in train + dev if d["domain"] == dom for l in d["labels"]) for dom in L.DOMAINS}
print(f"\n{'label':24s} " + " ".join(f"{dom:>8s}" for dom in L.DOMAINS) + "   (train + dev sentences)")
for l in L.LABELS:
    print(f"{l:24s} " + " ".join(f"{tab[dom][l]:8d}" for dom in L.DOMAINS))
tok = L.SentTokenizer()
n_tok = [len(tok.doc(d["sents"])[0]) for d in train + dev + test]
print(f"\ntokens per document: median {int(np.median(n_tok))}, max {max(n_tok)}; "
      f"tokens per sentence: {sum(n_tok) / sum(len(d['sents']) for d in train + dev + test):.1f}")
C = json.load(open("/kaggle/working/run_config_rr.json"))
for arch in sorted({C["experiments"][e]["arch"] for e in SESSION_EXPERIMENTS}):
    try:
        path = L.find_input(C["checkpoints"][arch])
        ck = torch.load(path, map_location="cpu", weights_only=False)
        print(f"checkpoint {arch:9s}: {path} | val_loss {ck.get('val_loss', float('nan')):.3f}")
    except Exception as e:
        print(f"checkpoint {arch:9s}: MISSING -> {e}")

## N9. CPU smoke test of the session's experiments

In [ ]:
# N9 — CPU smoke test (tiny random models, a few documents): every experiment must end with exit code 0.
import json, subprocess, sys
C = json.load(open("/kaggle/working/run_config_rr.json"))
smoke = []
for exp in SMOKE_EXPERIMENTS:
    t = C["experiments"][exp].get("teacher")
    smoke += [x for x in ([t] if t else []) + [exp] if x not in smoke]
for exp in smoke:
    r = subprocess.run([sys.executable, "train_rr.py", "--exp", exp, "--device", "cpu", "--smoke"],
                       capture_output=True, text=True, cwd="/kaggle/working")
    lines = [l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn(" not in l]
    print(f"===== {exp}: exit code {r.returncode}")
    print("\n".join(lines[-3:] if r.returncode == 0 else lines[-25:]))

## N10. Train the session's experiments on the GPUs + what each reached on its own

In [ ]:
# N10 — train SESSION_EXPERIMENTS: one experiment per GPU at a time; a distillation student waits for its teacher,
# a train_on_all run for its epochs_from run. Progress every 2 minutes.
import subprocess, sys, time, os, json, torch
C = json.load(open("/kaggle/working/run_config_rr.json"))
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
pending, running, finished = list(SESSION_EXPERIMENTS), {}, {}

def ready(exp):
    e = C["experiments"][exp]
    waits = [x for x in (e.get("teacher"), e.get("epochs_from")) if x]
    return all(t not in pending and t not in [r for r, _ in running.values()] for t in waits)

last_print = 0
while pending or running:
    for dev in devices:
        if dev in running and running[dev][1].poll() is not None:
            exp, p = running.pop(dev); finished[exp] = p.returncode
            print(time.strftime("%H:%M"), f"| {exp} finished on {dev} with exit code {p.returncode}", flush=True)
        if dev not in running:
            nxt = next((e for e in pending if ready(e)), None)
            if nxt:
                pending.remove(nxt)
                out = open(f"/kaggle/working/{nxt}.out", "w")
                running[dev] = (nxt, subprocess.Popen([sys.executable, "train_rr.py", "--exp", nxt, "--device", dev],
                                                      stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working"))
                print(time.strftime("%H:%M"), f"| started {nxt} on {dev}", flush=True)
    if time.time() - last_print > 120 and running:
        for dev, (exp, _) in running.items():
            lines = [l for l in open(f"/kaggle/working/{exp}.out").read().splitlines() if l.startswith(exp)]
            print(time.strftime("%H:%M"), "|", (lines[-1] if lines else f"{exp}: starting...")[:230], flush=True)
        last_print = time.time()
    time.sleep(15)
print("\nexit codes:", finished or "nothing trained this session")
for exp, code_ in finished.items():
    if code_ != 0:
        print(f"\n===== {exp} FAILED — last lines:\n" + open(f"/kaggle/working/{exp}.out").read()[-3000:])

In [ ]:
# N10b — what each run reached on its own (its best epoch, its own pooling, no post-processing)
import json, glob
rows = [json.load(open(p)) for p in sorted(glob.glob("/kaggle/working/rr_runs/*/summary.json"))]
print(f"{'run':24s} {'arch':9s} {'method':7s} {'train%':>7s} {'dev F1':>7s} {'CL':>6s} {'IT':>6s} {'ep':>3s} {'GPU GB':>6s}")
for s in sorted(rows, key=lambda s: -(s.get("dev_macro_f1") or -1)):
    f = lambda k: f"{s[k]:.3f}" if s.get(k) is not None else "  n/a"
    print(f"{s['exp']:24s} {s['arch']:9s} {s['method']:7s} {100 * s['params_trainable'] / s['params_total']:7.2f} "
          f"{f('dev_macro_f1'):>7s} {f('dev_f1_CL'):>6s} {f('dev_f1_IT'):>6s} {s['best_epoch']:3d} {str(s['peak_gpu_gb']):>6s}")

## N11. The lab — prediction sets, zips, index.csv, score chart

In [ ]:
# N11 — the lab (CPU, a few minutes): every finished run (this session's + earlier sessions added as input) x every
# pooling it supports x LAMBDAS x TAUS; your PROB_ENSEMBLES; greedy ensembles. Writes zips, index.csv, score chart.
import json, importlib, rr_common as L, rr_lab
importlib.reload(L); importlib.reload(rr_lab)
lab = rr_lab.Lab()
lab.load_runs(RUN_DIR_OVERRIDES)
lab.singles(LAMBDAS, TAUS)
lab.ensembles(PROB_ENSEMBLES, LAMBDAS, TAUS)
if GREEDY:
    lab.greedy(LAMBDAS, TAUS)
rows = lab.finish(TOP_N_ZIPS)
rr_lab.table(rows, 30)
rr_lab.chart(rows, "/kaggle/working/submissions/score_chart.png")
from IPython.display import Image, display
display(Image("/kaggle/working/submissions/score_chart.png"))

## N12. Check the zips + confirm the lab's scores with the official evaluator

In [ ]:
# N12 — check every zip (format, ids, sentence counts, labels) and confirm the lab's dev scores with the official
# evaluator (evaluate_phase3.py) on the 5 best written prediction sets.
import zipfile, json, glob, csv, subprocess, sys
import rr_common as L
_, _, test = L.load_data()
n_sent = {d["id"]: len(d["sents"]) for d in test}
zips = sorted(glob.glob("/kaggle/working/submissions/*.zip")) + sorted(glob.glob("/kaggle/working/rr_runs/*/submission_*.zip"))
bad_zips = 0
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    ok = (names == ["prediction.json"] and set(pred) == set(n_sent)
          and all(len(pred[k]) == n_sent[k] and all(l in L.L2I for l in pred[k]) for k in pred))
    bad_zips += not ok
    if not ok:
        print("PROBLEM", z)
print(f"{len(zips)} zips checked, {bad_zips} with problems")
V = "/kaggle/working/submissions/dev_predictions"
rows = [r for r in csv.DictReader(open("/kaggle/working/submissions/index.csv")) if r["file"].endswith(".zip") and r["dev_f1"]]
for r in rows[:5]:
    out = subprocess.run([sys.executable, "evaluate_phase3.py", "--truth-cl", f"{V}/dev_truth_CL.json", "--truth-it",
                          f"{V}/dev_truth_IT.json", "--prediction", f"{V}/{r['name']}.json", "--labels", f"{V}/labels.json"],
                         capture_output=True, text=True)
    off = [l.split()[-1] for l in out.stdout.splitlines() if l.startswith("Macro-F1")] or [out.stderr.strip()[-200:]]
    print(f"{r['name'][:62]:62s} lab {r['dev_f1']} | official {off[0]}")

## N13. Error analysis of one prediction set (material for the report)

In [ ]:
# N13 — error analysis of one written prediction set (material for the report): per-role F1 per domain and the most
# frequent confusions between roles.
import json, csv
from collections import Counter
import rr_common as L
NAME = None          # <- a name from submissions/index.csv; None = the best written one
V = "/kaggle/working/submissions/dev_predictions"
if NAME is None:
    NAME = next(r["name"] for r in csv.DictReader(open("/kaggle/working/submissions/index.csv"))
                if r["file"].endswith(".zip") and r["dev_f1"])
print("analysing:", NAME)
_, dev, _ = L.load_data()
pred = json.load(open(f"{V}/{NAME}.json"))
print(f"\n{'role':24s} {'F1 both':>8s} {'F1 CL':>7s} {'F1 IT':>7s} {'support':>8s}")
per = {dom: L.macro_scores([l for d in dev if d["domain"] == dom for l in d["labels"]],
                           [l for d in dev if d["domain"] == dom for l in pred[d["id"]]])["per_label"] for dom in L.DOMAINS}
allp = L.macro_scores([l for d in dev for l in d["labels"]], [l for d in dev for l in pred[d["id"]]])["per_label"]
for l in L.LABELS:
    print(f"{l:24s} {allp[l]['F1']:8.3f} {per['CL'][l]['F1']:7.3f} {per['IT'][l]['F1']:7.3f} {allp[l]['support']:8d}")
conf = Counter((g, p) for d in dev for g, p in zip(d["labels"], pred[d["id"]]) if g != p)
print("\nmost frequent confusions (gold -> predicted):")
for (g, p), c in conf.most_common(12):
    print(f"  {c:5d}  {g:24s} -> {p}")